# 04 · 事件传导预测模型

本 notebook 展示 Phase 4 模型的完整训练与分析流程：

1. 历史事件数据库查看
2. 特征分布探索 
3. 模型训练与 CV 评分
4. 特征重要性分析
5. SHAP 可解释性
6. 样本外预测 + 决策报告
7. 回测结果可视化

In [1]:
import sys, warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'PingFang SC'
plt.rcParams['axes.unicode_minus'] = False

repo_root = Path.cwd().parent
DATA_ROOT = repo_root / 'data'
print(f'Data root: {DATA_ROOT}')

Data root: /data00/home/guohuanwei.cztj/git_files/trade/data


## 1 · 历史事件数据库

In [2]:
from trade_py.db.event_db import EventDatabase, EventType

db = EventDatabase(DATA_ROOT)
events = db.events
print(f'Total events: {len(events)}')

if events:
    df_events = db.to_dataframe()
    display(df_events.head(10))
    fig, ax = plt.subplots(figsize=(10, 4))
    df_events['event_type'].value_counts().plot(kind='barh', ax=ax)
    ax.set_title('历史事件类型分布')
    ax.set_xlabel('事件数量')
    plt.tight_layout()
    plt.show()
else:
    print('No events yet. Run: trade model build-features')

Total events: 0
No events yet. Run: trade model build-features


## 2 · 特征数据探索

In [3]:
from trade_py.analysis.feature_builder import ALL_FEATURE_COLS, GROUP_A_COLS, GROUP_B_COLS, GROUP_C_COLS, GROUP_D_COLS

features_path = DATA_ROOT / 'events' / 'features.parquet'
labels_path   = DATA_ROOT / 'events' / 'propagation_labels.parquet'

if features_path.exists() and labels_path.exists():
    df_feat = pd.read_parquet(features_path)
    df_lbl  = pd.read_parquet(labels_path)
    df = df_feat.merge(df_lbl, on=['event_id', 'symbol'], how='inner')
    print(f'Feature rows: {len(df_feat):,}  |  Label rows: {len(df_lbl):,}  |  Merged: {len(df):,}')
    print(f'Feature columns: {len(ALL_FEATURE_COLS)}')
    display(df[GROUP_C_COLS].describe().round(4))
else:
    print('Features/labels not built yet.')
    df = pd.DataFrame()

Features/labels not built yet.


In [4]:
# Label distribution (if data available)
if not df.empty and 'return_20d' in df.columns:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, col in zip(axes, ['return_5d', 'return_20d', 'return_60d']):
        if col in df.columns:
            data = df[col].dropna()
            ax.hist(data, bins=50, edgecolor='none', alpha=0.7)
            ax.axvline(0, color='red', linestyle='--', linewidth=1)
            ax.set_title(f'{col}\nmean={data.mean():.3f}  std={data.std():.3f}')
            ax.set_xlabel('Excess Return')
    plt.suptitle('多时间窗口超额收益分布', y=1.02)
    plt.tight_layout()
    plt.show()

## 3 · 模型训练

In [ ]:
from trade_py.analysis.model_trainer import PropagationModel

model = PropagationModel(DATA_ROOT)

if features_path.exists() and labels_path.exists():
    model.load_data()
    cv_scores = model.train(n_cv_splits=5)
    print('\n=== CV Scores ===')
    for target, score in cv_scores.items():
        metric = 'IC' if 'return' in target else 'AUC'
        print(f'  {target:<25s} {metric}={score:.4f}')
    model.save(DATA_ROOT / 'models' / 'propagation')
else:
    print('Skipping training – data not built yet')
    print('Run:  trade model train')

## 4 · 特征重要性

In [ ]:
if (DATA_ROOT / 'models' / 'propagation' / 'return_20d.pkl').exists():
    if not model._models:  # load if not already trained in this session
        model.load(DATA_ROOT / 'models' / 'propagation')
    
    imp = model.feature_importance('return_20d', top_n=20)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(imp['feature'][::-1], imp['importance'][::-1])
    ax.set_title('Top 20 特征重要性（return_20d）')
    ax.set_xlabel('Importance (split gain)')
    plt.tight_layout()
    plt.show()
    display(imp)
else:
    print('Models not trained yet')

## 5 · SHAP 可解释性

In [ ]:
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print('shap not installed. Run: pip install shap')

if HAS_SHAP and not df.empty and model._models:
    feat_cols = model._feature_cols
    X_sample = df[feat_cols].fillna(0).head(200).to_numpy(dtype='float32')
    
    lgb_model = model._models.get('return_20d')
    if lgb_model is not None:
        explainer = shap.TreeExplainer(lgb_model)
        shap_vals = explainer.shap_values(X_sample)
        
        plt.figure(figsize=(10, 7))
        shap.summary_plot(shap_vals, X_sample, feature_names=feat_cols,
                          show=False, max_display=15)
        plt.title('SHAP Summary Plot: return_20d')
        plt.tight_layout()
        plt.show()

## 6 · 决策报告示例

In [ ]:
from datetime import date
from trade_py.db.event_db import HistoricalEvent, EventType, ActorType
from trade_py.analysis.feature_builder import FeatureBuilder
from trade_py.report.report_generator import ReportGenerator
from IPython.display import Markdown

sample_event = HistoricalEvent(
    event_date=date(2026, 2, 20),
    event_type=EventType.semiconductor_policy,
    magnitude=0.85,
    actor_type=ActorType.china_policy,
    primary_sector='SW_Electronics',
    breadth='sector',
    sentiment_score=0.75,
    news_volume=12,
    summary='国家大基金三期募集完成，重点支持先进半导体制程工艺研发',
)

DEMO_SYMBOL = '600703.SH'
DEMO_SECTOR = 'SW_Electronics'

if model._models:
    builder = FeatureBuilder(DATA_ROOT)
    feat_row = builder.build(sample_event, DEMO_SYMBOL, DEMO_SECTOR)
    if feat_row:
        gen = ReportGenerator(model)
        report = gen.generate(sample_event, DEMO_SYMBOL, feat_row.features, sector=DEMO_SECTOR)
        display(Markdown(gen.format_markdown(report)))
    else:
        print(f'No kline data for {DEMO_SYMBOL}')
else:
    print('Model not trained. Run cell 3 first.')

## 7 · 模拟事件驱动策略回测

In [ ]:
# Simple backtest: buy top-5 predicted return_20d stocks after each event,
# hold 20 trading days, compare to benchmark (000001.SH)

if not df.empty and 'return_20d' in df.columns and model._models:
    feat_cols = model._feature_cols
    X_all = df[feat_cols].fillna(0).to_numpy(dtype='float32')
    preds_20d = model._models['return_20d'].predict(X_all)
    
    df_bt = df[['date', 'symbol', 'return_20d']].copy()
    df_bt['pred_return_20d'] = preds_20d
    df_bt['date'] = pd.to_datetime(df_bt['date'])
    
    # Per event-date: pick top-5 symbols by prediction
    grouped = []
    for dt, g in df_bt.groupby('date'):
        top5 = g.nlargest(5, 'pred_return_20d')
        grouped.append({
            'date':          dt,
            'mean_actual':   top5['return_20d'].mean(),
            'mean_pred':     top5['pred_return_20d'].mean(),
            'n_selected':    len(top5),
        })
    
    bt_df = pd.DataFrame(grouped).sort_values('date').set_index('date')
    
    # Cumulative return
    bt_df['cum_actual'] = (1 + bt_df['mean_actual']).cumprod() - 1
    
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    
    # Top chart: cumulative excess return
    bt_df['cum_actual'].plot(ax=axes[0], color='steelblue', linewidth=2)
    axes[0].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[0].set_title('事件驱动策略：累计超额收益（Top-5 预测 × 20日持仓）')
    axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))
    axes[0].set_ylabel('累计超额收益')
    
    # Bottom chart: actual vs predicted per event
    axes[1].scatter(bt_df['mean_pred'], bt_df['mean_actual'], alpha=0.5, s=20)
    axes[1].axhline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[1].axvline(0, color='gray', linestyle='--', linewidth=0.8)
    axes[1].set_xlabel('预测超额收益')
    axes[1].set_ylabel('实际超额收益')
    axes[1].set_title('预测 vs 实际（每事件-日）')
    
    plt.tight_layout()
    plt.show()
    
    total_ret = bt_df['cum_actual'].iloc[-1] if len(bt_df) > 0 else 0
    from scipy.stats import pearsonr
    ic, _ = pearsonr(bt_df['mean_pred'], bt_df['mean_actual'])
    print(f'策略总超额收益: {total_ret*100:.1f}%')
    print(f'IC (预测 vs 实际): {ic:.4f}')
else:
    print('Need training data and model. Run cells 2-3 first.')

## 8 · 知识图谱传导路径可视化

In [ ]:
from trade_py.analysis.knowledge_graph import SectorGraph, SW_NAMES_ZH

sg = SectorGraph()
results = sg.propagate_event('semiconductor_policy', max_hop=2)

print('半导体政策事件传导路径（Top 15）:')
print(f'{"排名":<4} {"板块":<16} {"分数":>8} {"跳数":>6} {"时滞":>8} 传导路径')
print('-' * 80)
for i, r in enumerate(results[:15], 1):
    path_str = ' -> '.join(r.path[-3:])
    print(f'{i:<4} {r.sector_name:<16} {r.score:>+8.3f} {r.hop:>6}d {r.typical_days:>7}d  {path_str}')

In [ ]:
try:
    import networkx as nx
    from trade_py.analysis.knowledge_graph import SW, SW_NAMES_ZH, _EDGES

    G = nx.DiGraph()
    for sw in SW:
        G.add_node(sw.name, label=SW_NAMES_ZH.get(sw, sw.name))
    for edge in _EDGES:
        if edge.weight >= 0.60:
            G.add_edge(edge.source.name, edge.target.name, weight=edge.weight)

    pos = nx.spring_layout(G, k=1.5, seed=42)
    fig, ax = plt.subplots(figsize=(14, 10))
    labels = {n: G.nodes[n]['label'] for n in G.nodes}
    nx.draw_networkx_nodes(G, pos, node_size=500, node_color='steelblue', alpha=0.8, ax=ax)
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=7, ax=ax)
    nx.draw_networkx_edges(G, pos, alpha=0.4, arrows=True, arrowsize=10, ax=ax)
    ax.set_title('申万行业知识图谱（权重 ≥ 0.6 的边）')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    print(f'节点数: {G.number_of_nodes()}  边数: {G.number_of_edges()}')
except ImportError:
    print('networkx not installed')